In [3]:
import pandas as pd
import numpy as np
import os


SELECT p.photo_id, p.site_id, p.taken, s.site_name, s.latitude, s.longitude, c.species_id, c.prob, c.xmin, c.ymin, c.xmax, c.ymax FROM Classify c JOIN Photo p ON p.photo_id = c.photo_id LEFT JOIN Site s ON p.site_id = s.site_id WHERE c.origin = 'MEGA' AND c.species_id IN (87, 2077, 2049) AND p.status = 1 AND p.photo_id  ORDER BY `p`.`photo_id` DESC

In [6]:
supp_raw = pd.read_csv("supplement_raw_combined.csv")
mega = pd.read_csv(r"C:\Users\cheng\Desktop\dissertation1-main\initial_approach\mega_dataset.csv")
# convert species_id to numeric 
supp_raw["species_id"] = pd.to_numeric(supp_raw["species_id"], errors="coerce")
supp_raw["prob"] = pd.to_numeric(supp_raw["prob"], errors="coerce")

In [7]:
bbox_best = (
    supp_raw
    .dropna(subset=["species_id", "prob"])
    .sort_values(
        ["photo_id", "species_id", "prob"],
        ascending=[True, True, False]
    )
    .drop_duplicates(["photo_id", "species_id"], keep="first")
    .copy()
)

bbox_wide = bbox_best.pivot(
    index="photo_id",
    columns="species_id",
    values=["prob", "xmin", "ymin", "xmax", "ymax"]
)

bbox_wide.columns = [f"{v}_{int(s)}" for v, s in bbox_wide.columns]
bbox_wide = bbox_wide.reset_index()

bbox_wide = bbox_wide.rename(columns={
    "prob_87": "human_bbox_prob",
    "xmin_87": "human_xmin",
    "ymin_87": "human_ymin",
    "xmax_87": "human_xmax",
    "ymax_87": "human_ymax",

    "prob_2077": "animal_bbox_prob",
    "xmin_2077": "animal_xmin",
    "ymin_2077": "animal_ymin",
    "xmax_2077": "animal_xmax",
    "ymax_2077": "animal_ymax",

    "prob_2049": "vehicle_bbox_prob",
    "xmin_2049": "vehicle_xmin",
    "ymin_2049": "vehicle_ymin",
    "xmax_2049": "vehicle_xmax",
    "ymax_2049": "vehicle_ymax",
})

In [8]:
bbox_best[
    [
        "xmin",
        "ymin",
        "xmax",
        "ymax"
    ]
].describe()

,xmin,ymin,xmax,ymax
count,6.873939e+06,6.873939e+06,6.873939e+06,6.873939e+06
mean,8.908795e+02,5.908748e+02,1.575895e+03,8.652664e+02
std,6.447483e+02,4.885183e+02,8.051443e+02,5.410126e+02
min,0.000000e+00,0.000000e+00,1.244060e+01,3.375000e+00
25%,4.425330e+02,1.590620e+02,1.041650e+03,4.449790e+02
50%,8.674180e+02,5.387900e+02,1.459320e+03,7.896570e+02
75%,1.265510e+03,9.127940e+02,1.971540e+03,1.281600e+03
max,9.004030e+03,6.860040e+03,1.368510e+04,8.063190e+03


In [ ]:
print(supp_raw.shape)
print(supp_raw["photo_id"].nunique())

(20373401, 13)
6067033


In [ ]:
print(mega.shape)
print(mega["photo_id"].nunique())

(6720367, 10)
6720367


In [ ]:
mega_ids = set(mega["photo_id"])
supp_ids = set(supp_raw["photo_id"])

missing_ids = mega_ids - supp_ids

print("Missing photo_ids:", len(missing_ids))

Missing photo_ids: 653334


In [ ]:
list(sorted(missing_ids))[:20]

[162758,
 162763,
 162764,
 162765,
 162766,
 162767,
 162768,
 162769,
 162770,
 162775,
 162776,
 162777,
 162778,
 162779,
 162780,
 162781,
 162782,
 163357,
 163358,
 163359]

In [ ]:
mega_missing = mega[mega["photo_id"].isin(missing_ids)]

mega_missing.head(20)

,photo_id,sequence_id,sequence_num,contains_human,status,filename,dirname,mega_human_confidence,mega_animal_confidence,mega_vehicle_confidence
0,162758,27166360,1,0,1,10869dafe4d86eff611698ecadd505ce.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
1,162763,27166360,2,0,1,cffc7732946fab09f665f5ca8d9f0e8e.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
2,162764,27166360,3,0,1,3029a548a17595338aa26581720920ea.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
3,162765,27166359,1,0,1,480425e2f1540e1176c219d323fa78ef.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
4,162766,27166360,4,0,1,119eb4d6873d2dc6ca931a40957f74b5.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
5,162767,27166361,1,0,1,b3defd0b0f1f78d436a1bc5a5d7d7413.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
6,162768,27166361,3,0,1,7337a5544041a436cb86965953802727.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
7,162769,27166361,2,0,1,aa65c6f086b057fa6944973b3296cb7c.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
8,162770,27166361,4,0,1,5f2fa7d0cc6d817c4887e598bd17e469.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0
9,162775,27166363,1,0,1,f6ee2e582cf4a0e94dc22e54785b276f.jpg,/var/www/html/biodivimages/person_816/site_345,0.0,0.0,0.0


In [ ]:
mega_missing[
    [
        "photo_id",
        "mega_human_confidence",
        "mega_animal_confidence",
        "mega_vehicle_confidence"
    ]
].head(20)

,photo_id,mega_human_confidence,mega_animal_confidence,mega_vehicle_confidence
0,162758,0.0,0.0,0.0
1,162763,0.0,0.0,0.0
2,162764,0.0,0.0,0.0
3,162765,0.0,0.0,0.0
4,162766,0.0,0.0,0.0
5,162767,0.0,0.0,0.0
6,162768,0.0,0.0,0.0
7,162769,0.0,0.0,0.0
8,162770,0.0,0.0,0.0
9,162775,0.0,0.0,0.0


In [ ]:
(
    mega_missing[
        [
            "mega_human_confidence",
            "mega_animal_confidence",
            "mega_vehicle_confidence"
        ]
    ] > 0
).any(axis=1).value_counts()

False    653130
True        204
Name: count, dtype: int64

In [ ]:
mega_missing_positive = mega_missing[
    (
        mega_missing[
            [
                "mega_human_confidence",
                "mega_animal_confidence",
                "mega_vehicle_confidence"
            ]
        ] > 0
    ).any(axis=1)
]

mega_missing_positive.head(20)

,photo_id,sequence_id,sequence_num,contains_human,status,filename,dirname,mega_human_confidence,mega_animal_confidence,mega_vehicle_confidence
1670318,14835203,31144706,1,0,1,2f9ac74e36d85880677e95df374d3b31.jpg,/var/www/html/biodivimages/person_16303/site_6449,0.0,0.9750,0.0
1670319,14835204,31144706,2,0,1,33a813af65c7c0153621790ed83d7cb8.jpg,/var/www/html/biodivimages/person_16303/site_6449,0.0,0.9750,0.0
1670320,14835205,31144706,3,0,1,2a280400699452498eff2331ecbf356e.jpg,/var/www/html/biodivimages/person_16303/site_6449,0.0,0.6190,0.0
1799436,15118353,31208915,1,0,1,b635947f82e1aaff99cf4dcc149c6f3f.jpg,/var/www/html/biodivimages/person_18727/site_6329,0.0,0.2000,0.0
2520236,17354129,31696084,1,0,1,ccf5bf73bbb18794d1fd8d82261ef53e.jpg,/var/www/html/biodivimages/person_22690/site_6798,0.0,0.4330,0.0
2520237,17354131,31696084,2,0,1,2dfee2d8a887aaf0369259c95cc5a9ea.jpg,/var/www/html/biodivimages/person_22690/site_6798,0.0,0.3200,0.0
2520238,17354134,31696084,3,0,1,8f40066b4fc6c47e06b45f7a59666966.jpg,/var/www/html/biodivimages/person_22690/site_6798,0.0,0.0403,0.0
2561697,17520750,31730299,1,0,1,227f5f08ef23e6a75e8f502572966cbf.jpg,/var/www/html/biodivimages/person_22690/site_6825,0.0,0.9040,0.0
2788102,18336347,31877834,2,0,1,d961b24ae74f59f74da22244944b7a7a.jpg,/var/www/html/biodivimages/person_22385/site_6952,0.0,0.9350,0.0
2788103,18336348,31877834,1,0,1,18e007233bc8819800576d9ee7499af1.jpg,/var/www/html/biodivimages/person_22385/site_6952,0.0,0.9380,0.0


In [ ]:
mega_missing_positive[
    [
        "mega_human_confidence",
        "mega_animal_confidence",
        "mega_vehicle_confidence"
    ]
].describe()

,mega_human_confidence,mega_animal_confidence,mega_vehicle_confidence
count,204.000000,204.000000,204.000000
mean,0.001391,0.464088,0.028934
std,0.007798,0.386643,0.149058
min,0.000000,0.000000,0.000000
25%,0.000000,0.083550,0.000000
50%,0.000000,0.330500,0.000000
75%,0.000000,0.917250,0.000000
max,0.063600,0.977000,0.924000


In [ ]:
mega["missing_from_supp"] = mega["photo_id"].isin(missing_ids)
mega["missing_positive_from_supp"] = mega["photo_id"].isin(mega_missing_positive["photo_id"])

In [ ]:
new_mega = mega.merge(
    supp_photo_level,
    on="photo_id",
    how="left"
)

for col in ["n_human_boxes", "n_animal_boxes", "n_vehicle_boxes"]:
    new_mega[col] = new_mega[col].fillna(0).astype(int)



In [ ]:
print("Shape:", new_mega.shape)
print()

print("Columns:")
print(new_mega.columns.tolist())
print()

new_mega.info()

Shape: (6720367, 35)

Columns:
['photo_id', 'sequence_id', 'sequence_num', 'contains_human', 'status', 'filename', 'dirname', 'mega_human_confidence', 'mega_animal_confidence', 'mega_vehicle_confidence', 'missing_from_supp', 'missing_positive_from_supp', 'site_id', 'taken', 'site_name', 'latitude', 'longitude', 'n_human_boxes', 'n_vehicle_boxes', 'n_animal_boxes', 'human_bbox_prob', 'vehicle_bbox_prob', 'animal_bbox_prob', 'human_xmin', 'vehicle_xmin', 'animal_xmin', 'human_ymin', 'vehicle_ymin', 'animal_ymin', 'human_xmax', 'vehicle_xmax', 'animal_xmax', 'human_ymax', 'vehicle_ymax', 'animal_ymax']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6720367 entries, 0 to 6720366
Data columns (total 35 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   photo_id                    int64  
 1   sequence_id                 int64  
 2   sequence_num                int64  
 3   contains_human              int64  
 4   status                      

In [ ]:
new_mega.tail()

,photo_id,sequence_id,sequence_num,contains_human,status,filename,dirname,mega_human_confidence,mega_animal_confidence,mega_vehicle_confidence,...,animal_xmin,human_ymin,vehicle_ymin,animal_ymin,human_xmax,vehicle_xmax,animal_xmax,human_ymax,vehicle_ymax,animal_ymax
6720362,10999958,30225544,2,0,1,c529843eaccccd929ca16dd4b07093b7.jpg,/var/www/html/biodivimages/person_20667/site_5552,0.0000,0.0000,0.0336,...,NaN,NaN,922.968,NaN,NaN,1068.44,NaN,NaN,1079.89,NaN
6720363,10999959,30225901,1,0,1,89f6efa9dd1234b517cbd582aeb4f08e.jpg,/var/www/html/biodivimages/person_3221/site_5503,0.0000,0.0243,0.0244,...,1180.30,NaN,1200.230,0.0,NaN,1109.07,1932.13,NaN,1511.85,51.7406
6720364,10999960,30225544,3,0,1,38bdde445261b4fa5dfd19cdcefe7f0c.jpg,/var/www/html/biodivimages/person_20667/site_5552,0.0152,0.0000,0.0741,...,NaN,304.776,919.620,NaN,600.768,1055.96,NaN,327.834,1079.89,NaN
6720365,10999962,30225901,2,0,1,e8fa201b5a6050e324925edc63a8481d.jpg,/var/www/html/biodivimages/person_3221/site_5503,0.0000,0.1860,0.0000,...,1717.09,NaN,NaN,0.0,NaN,NaN,2117.07,NaN,NaN,67.4957
6720366,10999967,30225901,3,0,1,36bd8bef6823ac13529047107bb7b695.jpg,/var/www/html/biodivimages/person_3221/site_5503,0.0000,0.0719,0.0000,...,1720.86,NaN,NaN,0.0,NaN,NaN,2125.94,NaN,NaN,66.3617


In [ ]:
# Missing values in the newly added columns
new_cols = [
    "site_id",
    "taken",
    "site_name",
    "latitude",
    "longitude",
    "human_xmin",
    "animal_xmin",
    "vehicle_xmin"
]

new_mega[new_cols].isna().sum()

site_id          653334
taken            653334
site_name        653334
latitude         653334
longitude        653334
human_xmin      6105814
animal_xmin     1140903
vehicle_xmin    6040445
dtype: int64

In [ ]:
# Check bbox counts
new_mega[
    [
        "n_human_boxes",
        "n_animal_boxes",
        "n_vehicle_boxes"
    ]
].describe()

,n_human_boxes,n_animal_boxes,n_vehicle_boxes
count,6.720367e+06,6.720367e+06,6.720367e+06
mean,1.574569e-01,2.660719e+00,2.134151e-01
std,7.155273e-01,5.783425e+00,8.942712e-01
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,1.000000e+00,0.000000e+00
50%,0.000000e+00,1.000000e+00,0.000000e+00
75%,0.000000e+00,2.000000e+00,0.000000e+00
max,7.500000e+01,3.080000e+02,3.000000e+01


In [ ]:
new_mega.to_csv("new_mega.csv", index=False)

In [ ]:
supp_photo_level.to_csv("supp_photo_level.csv", index=False)

In [ ]:
print("supp_raw:", supp_raw.shape)
print("supp_photo_level:", supp_photo_level.shape)
print("new_mega:", new_mega.shape)

print("Unique photo_ids in new_mega:",
      new_mega["photo_id"].nunique())

print("Duplicate photo_ids:",
      new_mega["photo_id"].duplicated().sum())

supp_raw: (20373401, 13)
supp_photo_level: (6067033, 24)
new_mega: (6720367, 35)
Unique photo_ids in new_mega: 6720367
Duplicate photo_ids: 0


chunk1.csv
...
chunk24.csv
        ↓
supplement_raw_combined.csv   ← RAW (keep exactly as merged)
        ↓
supp_photo_level.csv          ← processed
        ↓
new_mega.csv                  ← final merged dataset